
# ILLM Lab 5 — Word2Vec, FastText, Doc2Vec, K-Means Word Clusters & Nearest-Sentence Retrieval

Uses the **same 1,000,000-sentence Hindi corpus and the same train/validation/test
split** as Labs 3 and 4. Trains three embedding models with `gensim`
(following the official tutorials the assignment links to), clusters the
vocabulary with K-means, and evaluates all the embeddings by using them to find,
for every validation/test sentence, its nearest neighbor in the training set.

1. Train **Word2Vec**, **FastText**, and **Doc2Vec** on the training data.
2. **K-means** cluster the vocabulary; show the 20 words closest to each centroid.
3. Represent every validation/test sentence as a vector — via averaged word
   embeddings *and* via Doc2Vec — and retrieve its nearest training sentence.

---

## 1. Deep analysis of the assignment

| # | Requirement | What it means concretely |
|---|---|---|
| 1a | Train Word2Vec on the training data | `gensim.models.Word2Vec`, following [the official tutorial](https://radimrehurek.com/gensim/auto_examples/tutorials/run_word2vec.html): one dense vector per vocabulary word, learned from local word co-occurrence. |
| 1b | Train FastText on the training data | `gensim.models.FastText`, following [the official tutorial](https://radimrehurek.com/gensim/auto_examples/tutorials/run_fasttext.html): like Word2Vec, but a word's vector is built from its character n-grams, so it can embed words never seen in training (this matters a lot for a morphologically rich language like Hindi). |
| 1c | Train Doc2Vec on the training data | `gensim.models.doc2vec.Doc2Vec`, following [the official tutorial](https://radimrehurek.com/gensim/auto_examples/tutorials/run_doc2vec_lee.html): every sentence gets its own trained vector (a "paragraph vector"), not just an average of its words' vectors. |
| 2a | K-means cluster the words; show 20 words closest to each centroid | Run `sklearn.cluster.KMeans` directly on the trained word-vector matrix; for every centroid, rank *all* vocabulary words by distance to it and keep the top 20 — not just the words K-means happened to assign to that cluster (those two lists nearly coincide, but "closest to the centroid" is the literal instruction). |
| 2b | Represent val/test sentences as vectors (word embeddings **and** Doc2Vec); find the closest sentence | For every validation and test sentence: build (i) an averaged-word-embedding vector and (ii) a Doc2Vec-inferred vector, then retrieve its nearest neighbor **among the training sentences** by cosine similarity, under each representation. |

**Design decisions (stated explicitly, since the assignment doesn't specify these):**

- **Same corpus, same splits, same tokenization as Labs 3–4.** Train = reload of
  the exact same `train_pool` from Lab 3; validation = Lab 3/4's `dev_sentences`
  (the assignment says "validation set", Labs 3–4 called it "dev" — same set);
  test = the same `test_sentences`. Tokenization is the same lowercase
  whitespace split used in Lab 4, for consistency across the whole course project.
- **"Word embeddings" sentence vectors, plural.** Since the assignment trained
  *both* Word2Vec and FastText, both are used to build averaged sentence vectors
  for the nearest-neighbor task (not just one) — this also makes it possible to
  show FastText's key practical advantage (handling words never seen in
  training) directly in the results, rather than only asserting it.
- **Averaging rule:** a sentence vector is the mean of its in-vocabulary words'
  vectors. For Word2Vec, a word absent from the trained vocabulary is skipped
  entirely, since Word2Vec has no way to embed it (Lab 4-style `<UNK>`
  replacement is not used here, since gensim's own vocabulary handling already
  covers this). For FastText, every word contributes a vector — via character
  n-grams — even if the exact word was never seen in training.
- **K-means cluster count $K$** is not specified by the assignment. $K=50$ is
  used as a reasonable default for a vocabulary of tens of thousands of words;
  this is easy to change (see the `K_CLUSTERS` variable in Section 8).
- **Doc2Vec epochs:** the official tutorial uses 40 epochs, but that is for a
  314-document toy corpus. At 1,000,000 documents, 40 epochs would take far too
  long; 10 epochs is used instead, as a stated, explicit tradeoff between
  training time and vector quality.
- **Nearest-neighbor search is done with vectorized cosine similarity**
  ($L_2$-normalize every vector, then one matrix multiplication), not a
  Python loop comparing one pair at a time — this is the difference between a
  search that finishes in seconds and one that doesn't finish at all at
  1,000,000-sentence scale.

## 2. Plan for this notebook

```
Reload train/validation/test sentences from Lab 3 (same corpus, same splits)
        ▼
Tokenize (same lowercase + whitespace split as Lab 4)
        ▼
Train Word2Vec  -->  Train FastText  -->  Train Doc2Vec   (Task 1, a/b/c)
        ▼
K-means on the Word2Vec vocabulary; 20 nearest words per centroid  (Task 2a)
        ▼
Build averaged-word-embedding vectors (Word2Vec & FastText) for every
validation/test sentence, and infer Doc2Vec vectors for them            (Task 2b)
        ▼
Vectorized nearest-neighbor search against all training sentences,
for each of the 3 representations, on both validation and test
        ▼
Compare & discuss
```



## Section 1 — Environment Setup


In [1]:

import os
import time
import random
import pickle

import numpy as np
import pandas as pd

from gensim.models import Word2Vec, FastText
from gensim.models.doc2vec import Doc2Vec, TaggedDocument
from sklearn.cluster import KMeans

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

LANGUAGE_CODE = "hin_Deva"   # must match the value used in Labs 3-4
VECTOR_SIZE = 100
WINDOW = 5
MIN_COUNT = 5
WORKERS = os.cpu_count() or 4
EPOCHS_W2V_FT = 5
EPOCHS_D2V = 10
K_CLUSTERS = 50

print(f"Using {WORKERS} worker threads.")
print("Environment ready.")


Using 8 worker threads.
Environment ready.



## Section 2 — Reload the SAME Corpus & Splits from Lab 3

Identical approach to Lab 4, Section 2: reload the exact sentences Lab 3 saved to
disk, rather than re-deriving or re-downloading them, so the split here is
byte-identical to every previous lab.

> **If you see a `FileNotFoundError` here:** run the Lab 3 notebook first
> (through at least its Section 13), in the same working directory as this
> notebook.


In [2]:

CSV_PATH = f"/Users/akshdharodiya/Desktop/SEM_7_LAB_PRACTICALS/LLM_LAB/LLM_LAB_3_13_08_26/hin_Deva_sentences.csv"
PKL_PATH = f"/Users/akshdharodiya/Desktop/SEM_7_LAB_PRACTICALS/LLM_LAB/LLM_LAB_3_13_08_26/hin_Deva_tokenizer_results.pkl"

for path in (CSV_PATH, PKL_PATH):
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Missing '{path}'. Run the Lab 3 (BPE/WordPiece) notebook first, in this "
            f"same working directory, so the corpus + splits it saved are available here."
        )

all_sentences = pd.read_csv(CSV_PATH, encoding="utf-8")["text"].tolist()

with open(PKL_PATH, "rb") as f:
    prev_results = pickle.load(f)
val_sentences_raw = prev_results["dev_sentences"]     # "validation" in this assignment's wording
test_sentences_raw = prev_results["test_sentences"]

_val_set = set(val_sentences_raw)
_test_set = set(test_sentences_raw)
train_sentences_raw = [s for s in all_sentences if s not in _val_set and s not in _test_set]

print(f"Reloaded from Lab 3:")
print(f"  Train      : {len(train_sentences_raw):,} sentences")
print(f"  Validation : {len(val_sentences_raw):,} sentences")
print(f"  Test       : {len(test_sentences_raw):,} sentences")


Reloaded from Lab 3:
  Train      : 1,000,000 sentences
  Validation : 1,000 sentences
  Test       : 1,000 sentences



## Section 3 — Tokenization

Same rule as Lab 4: lowercase, then split on whitespace.


In [3]:

def word_tokenize(sentence):
    return sentence.lower().split()

train_tokens = [word_tokenize(s) for s in train_sentences_raw]
val_tokens = [word_tokenize(s) for s in val_sentences_raw]
test_tokens = [word_tokenize(s) for s in test_sentences_raw]

print("Example tokenized training sentence:")
print(train_tokens[0])
print(f"\nTotal training tokens: {sum(len(s) for s in train_tokens):,}")


Example tokenized training sentence:
['लोगों', 'को', 'बिलों', 'संबंधी', 'सुविधा', 'देना', 'ही', 'उनका', 'काम']

Total training tokens: 17,995,974



## Section 4 — Task 1a: Train Word2Vec

Following [the official Word2Vec tutorial](https://radimrehurek.com/gensim/auto_examples/tutorials/run_word2vec.html):
train directly on the tokenized sentences, with the tutorial's documented
defaults except where noted:

- `vector_size=100` — dimensionality of each word vector.
- `window=5` — how many neighboring words count as "context".
- `min_count=5` *(the tutorial's own documented default)* — ignore words seen
  fewer than 5 times; too rare to learn a meaningful vector for anyway.
- `sg=0` (CBOW) — predict the center word from the average of its context
  words. CBOW trains faster than skip-gram and works well for frequent words,
  a reasonable default at this corpus size; skip-gram (`sg=1`) is the more
  common choice specifically when rare-word quality matters more than speed.
- `epochs=5` — the tutorial's own default pass count.

Expect this cell to take a few minutes on the full 1,000,000-sentence training set.


In [4]:

print("Training Word2Vec...")
t0 = time.time()
w2v_model = Word2Vec(
    sentences=train_tokens,
    vector_size=VECTOR_SIZE,
    window=WINDOW,
    min_count=MIN_COUNT,
    workers=WORKERS,
    sg=0,
    epochs=EPOCHS_W2V_FT,
    seed=RANDOM_SEED,
)
print(f"Word2Vec trained in {time.time()-t0:.1f}s")
print(f"Vocabulary size: {len(w2v_model.wv.index_to_key):,}")


Training Word2Vec...


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


Word2Vec trained in 23.5s
Vocabulary size: 78,251


In [5]:

# Quick qualitative sanity check, same spirit as the tutorial's own demo section:
# do nearby words in vector space actually look related?
sample_words = [w for w in ["पानी", "स्कूल", "अच्छा", "भारत", "काम"] if w in w2v_model.wv.key_to_index]
if not sample_words:
    # fallback: just take a few of the most frequent vocabulary words
    sample_words = w2v_model.wv.index_to_key[:5]

for w in sample_words:
    print(f"Word2Vec most similar to '{w}':")
    for sim_word, score in w2v_model.wv.most_similar(w, topn=5):
        print(f"    {sim_word:<15} {score:.3f}")
    print()


Word2Vec most similar to 'पानी':
    मिट्टी          0.702
    दूध             0.692
    कचरे            0.683
    बर्फ            0.680
    नालों           0.673

Word2Vec most similar to 'स्कूल':
    कॉलेज           0.772
    स्‍कूल          0.765
    विद्यालय        0.758
    हाईस्कूल        0.749
    हॉस्टल          0.744

Word2Vec most similar to 'अच्छा':
    अच्‍छा          0.783
    बढ़िया          0.779
    अच्छे           0.748
    बेहतर           0.717
    बेहतरीन         0.677

Word2Vec most similar to 'भारत':
    अमेरिका         0.765
    चीन             0.762
    देश             0.749
    जापान           0.747
    रूस             0.723

Word2Vec most similar to 'काम':
    कार्य           0.747
    प्रयास          0.636
    काम-काज         0.592
    प्रयत्न         0.577
    खर्च            0.573




## Section 5 — Task 1b: Train FastText

Following [the official FastText tutorial](https://radimrehurek.com/gensim/auto_examples/tutorials/run_fasttext.html).
Same core hyperparameters as Word2Vec (so the two are comparable), plus
FastText's own defaults for character n-gram length: `min_n=3`, `max_n=6` — each
word's vector is the sum of its 3-to-6-character n-gram vectors, which is what
lets FastText embed a word it never saw during training, as long as it shares
some n-grams with words it did see.


In [6]:

print("Training FastText...")
t0 = time.time()
ft_model = FastText(
    sentences=train_tokens,
    vector_size=VECTOR_SIZE,
    window=WINDOW,
    min_count=MIN_COUNT,
    workers=WORKERS,
    sg=0,
    epochs=EPOCHS_W2V_FT,
    seed=RANDOM_SEED,
)
print(f"FastText trained in {time.time()-t0:.1f}s")
print(f"Vocabulary size: {len(ft_model.wv.index_to_key):,}")


Training FastText...


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


FastText trained in 52.8s
Vocabulary size: 78,251


In [7]:

# Demonstrate FastText's headline advantage over Word2Vec: an out-of-vocabulary
# word (one that never occurred often enough in training to get its own slot)
# still gets a usable vector, built from its character n-grams.
oov_candidates = []
for s in val_tokens[:200]:
    for w in s:
        if w not in w2v_model.wv.key_to_index:
            oov_candidates.append(w)
oov_candidates = list(dict.fromkeys(oov_candidates))  # de-duplicate, keep order

if oov_candidates:
    example_oov = oov_candidates[0]
    print(f"Example word absent from Word2Vec's vocabulary: '{example_oov}'")
    try:
        w2v_model.wv[example_oov]
        print("  Word2Vec: (unexpectedly succeeded)")
    except KeyError:
        print("  Word2Vec: KeyError -- cannot embed this word at all.")
    ft_vec = ft_model.wv[example_oov]
    print(f"  FastText: produced a {ft_vec.shape[0]}-dim vector anyway (via character n-grams).")
else:
    print("No OOV words found in the first 200 validation sentences (vocabulary coverage is high).")


Example word absent from Word2Vec's vocabulary: '(जी.एन.एस.)'
  Word2Vec: KeyError -- cannot embed this word at all.
  FastText: produced a 100-dim vector anyway (via character n-grams).



## Section 6 — Task 1c: Train Doc2Vec

Following [the official Doc2Vec tutorial](https://radimrehurek.com/gensim/auto_examples/tutorials/run_doc2vec_lee.html).
Every training sentence needs a `TaggedDocument` with a unique tag — its
position in the training list is used, exactly as the tutorial does for the
Lee corpus.

**Epochs, explicitly justified:** the tutorial trains for 40 epochs, but that is
for a 314-document *toy* corpus, where more passes are cheap and small-data
overfitting isn't a concern for a demo. At 1,000,000 documents, 40 epochs would
multiply the already-substantial per-epoch cost eight-fold for comparatively
little extra benefit — `epochs=10` is used instead as an explicit,
stated time/quality tradeoff.


In [8]:

train_tagged = [TaggedDocument(words=tokens, tags=[i]) for i, tokens in enumerate(train_tokens)]

print("Building Doc2Vec vocabulary...")
d2v_model = Doc2Vec(
    vector_size=VECTOR_SIZE,
    min_count=MIN_COUNT,
    workers=WORKERS,
    epochs=EPOCHS_D2V,
    seed=RANDOM_SEED,
)
d2v_model.build_vocab(train_tagged)

print("Training Doc2Vec...")
t0 = time.time()
d2v_model.train(train_tagged, total_examples=d2v_model.corpus_count, epochs=d2v_model.epochs)
print(f"Doc2Vec trained in {time.time()-t0:.1f}s")
print(f"Vocabulary size: {len(d2v_model.wv.index_to_key):,}")
print(f"Document vectors: {len(d2v_model.dv):,}")


Building Doc2Vec vocabulary...
Training Doc2Vec...


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


Doc2Vec trained in 300.0s
Vocabulary size: 78,251
Document vectors: 1,000,000


In [9]:

# Same self-similarity sanity check the tutorial itself uses: re-infer a vector
# for a handful of TRAINING documents and confirm the model considers each one
# most similar to itself (a basic check that training actually worked, not a
# real accuracy measure -- inferred vectors are stochastic, so this is 'mostly
# itself' rather than 'always itself').
sample_ids = random.sample(range(len(train_tagged)), 20)
self_rank_hits = 0
for doc_id in sample_ids:
    inferred = d2v_model.infer_vector(train_tagged[doc_id].words)
    sims = d2v_model.dv.most_similar([inferred], topn=1)
    if sims[0][0] == doc_id:
        self_rank_hits += 1
print(f"Self-similarity sanity check: {self_rank_hits}/{len(sample_ids)} sampled training "
      f"documents were most similar to themselves after re-inference.")


Self-similarity sanity check: 10/20 sampled training documents were most similar to themselves after re-inference.



## Section 7 — Task 2a: K-Means Clustering of the Vocabulary

Clusters the **Word2Vec** word vectors directly with `sklearn.cluster.KMeans`
into $K=50$ clusters (`K_CLUSTERS`, Section 1 — change it there to try a
different value).

For each of the $K$ cluster centroids, every vocabulary word is ranked by
Euclidean distance to that centroid, and the **20 closest** are kept — this is
the literal reading of "represent each cluster with 20 words closest to each
cluster centroid": nearest to the *centroid point*, not simply "whichever
words K-means happened to assign to that cluster" (in practice the two lists
are almost always the same words, since K-means assigns each point to its
nearest centroid by construction — but a handful of borderline words can
differ, and computing it this way answers exactly what was asked).


In [10]:

words = w2v_model.wv.index_to_key
word_vectors = w2v_model.wv.vectors  # shape (vocab_size, VECTOR_SIZE), in the same order as `words`

print(f"Clustering {len(words):,} vocabulary words into {K_CLUSTERS} clusters...")
t0 = time.time()
kmeans = KMeans(n_clusters=K_CLUSTERS, random_state=RANDOM_SEED, n_init=10).fit(word_vectors)
print(f"K-means finished in {time.time()-t0:.1f}s")


Clustering 78,251 vocabulary words into 50 clusters...
K-means finished in 9.6s


In [11]:

def closest_words_to_centroid(centroid, words, word_vectors, top_n=20):
    dists = np.linalg.norm(word_vectors - centroid, axis=1)
    nearest_idx = np.argsort(dists)[:top_n]
    return [(words[i], dists[i]) for i in nearest_idx]

cluster_sizes = np.bincount(kmeans.labels_, minlength=K_CLUSTERS)

for c in range(K_CLUSTERS):
    nearest = closest_words_to_centroid(kmeans.cluster_centers_[c], words, word_vectors, top_n=20)
    nearest_words = [w for w, _ in nearest]
    print(f"Cluster {c:2d} ({cluster_sizes[c]:,} words assigned) -- 20 closest to centroid:")
    print(f"    {', '.join(nearest_words)}")
    print()


Cluster  0 (1,396 words assigned) -- 20 closest to centroid:
    मोहनलाल, भाटिया,, सेन,, पल्लवी, रमेश,, परिहार,, वासुदेव, बद्री, पहलवान,, पंडित,, ओझा,, हरिओम, त्रिलोक, मेहरा,, गंगवार,, ध्रुव,, सोनू,, विनीता, कामरेड, रेनू

Cluster  1 (340 words assigned) -- 20 closest to centroid:
    संस्कारों, अहिंसा, कथाओं, ग्रन्थ, ज्ञान,, मनुष्यों, साधक, यथार्थ, अध्यात्म, परमेश्वर, लघुकथा, ब्रह्मांड, पंथ, सहिष्णुता, वेदों, परमात्मा, अर्थात्, बोध, चेतना, दार्शनिक

Cluster  2 (2,650 words assigned) -- 20 closest to centroid:
    आईपैड, स्पीकर्स, पैनासोनिक, हेडलैंप,, स्‍टोरेज, डिस्पले, abs, oled, स्क्रीन,, जीपीएस,, टेक्नॉलजी, पोर्ट्स, लार्ज, आसुस, क्लस्टर,, मैकबुक, g, ऑनबोर्ड, एलइडी, सिल्वर,

Cluster  3 (167 words assigned) -- 20 closest to centroid:
    संभागीय, प्रकोष्ठ, प्रांतीय, उपमंडल, पर्यवेक्षक, गढ़वाल, नगरपालिका, संकुल, पदाधिकारी,, अध्यक्ष,, न्यास, उपखण्ड, खण्ड, अंचल, नवनियुक्त, सांख्यिकी, कार्यालय,, कनिष्ठ, कोरबा, समिति,

Cluster  4 (221 words assigned) -- 20 closest to centroid:
    बाईपास, फ


## Section 8 — Task 2b: Sentence Vectors & Nearest-Neighbor Retrieval

Three sentence representations are built and compared:

1. **Word2Vec-averaged** — mean of the Word2Vec vectors of a sentence's
   in-vocabulary words (words never seen in training are skipped — Word2Vec has
   no way to embed them).
2. **FastText-averaged** — the same averaging, but using FastText, which can
   embed *every* word via character n-grams, including ones never seen in
   training.
3. **Doc2Vec-inferred** — `model.infer_vector(tokens)`, which does not average
   anything; it runs a few steps of the same training-style optimization to
   find the paragraph vector that best explains this specific sequence of
   words, holding the trained word vectors fixed.

For every validation and test sentence, under each representation, its
**nearest neighbor among all training sentences** is retrieved by cosine
similarity.

$$\text{cosine\_sim}(u, v) = \frac{u \cdot v}{\lVert u \rVert \, \lVert v \rVert}$$

**Why a matrix multiplication, not a loop:** with ~1,000,000 training
sentences and 2,000 validation+test sentences, a naive "compare every pair
one at a time" loop is 2,000,000,000 individual comparisons — far too slow in
pure Python. $L_2$-normalizing every vector first turns cosine similarity into
a plain dot product, so **all** similarities between a batch of query vectors
and every training vector can be computed in a single matrix multiplication,
letting optimized BLAS routines do the work instead of the Python interpreter.


In [12]:

def average_vector(tokens, keyed_vectors):
    vecs = [keyed_vectors[w] for w in tokens if w in keyed_vectors.key_to_index]
    if not vecs:
        return np.zeros(keyed_vectors.vector_size, dtype=np.float32)
    return np.mean(vecs, axis=0).astype(np.float32)

def l2_normalize_rows(matrix):
    norms = np.linalg.norm(matrix, axis=1, keepdims=True)
    norms[norms == 0] = 1.0  # avoid dividing an all-zero (fully-OOV) sentence vector by zero
    return matrix / norms

def build_vectors(token_lists, keyed_vectors, label):
    t0 = time.time()
    vecs = np.array([average_vector(toks, keyed_vectors) for toks in token_lists], dtype=np.float32)
    print(f"  {label}: {vecs.shape[0]:,} vectors built in {time.time()-t0:.1f}s")
    return vecs


In [13]:

print("Building Word2Vec-averaged sentence vectors...")
train_w2v_vecs = build_vectors(train_tokens, w2v_model.wv, "train")
val_w2v_vecs = build_vectors(val_tokens, w2v_model.wv, "validation")
test_w2v_vecs = build_vectors(test_tokens, w2v_model.wv, "test")

print("\nBuilding FastText-averaged sentence vectors...")
train_ft_vecs = build_vectors(train_tokens, ft_model.wv, "train")
val_ft_vecs = build_vectors(val_tokens, ft_model.wv, "validation")
test_ft_vecs = build_vectors(test_tokens, ft_model.wv, "test")


Building Word2Vec-averaged sentence vectors...
  train: 1,000,000 vectors built in 13.5s
  validation: 1,000 vectors built in 0.0s
  test: 1,000 vectors built in 0.0s

Building FastText-averaged sentence vectors...
  train: 1,000,000 vectors built in 15.3s
  validation: 1,000 vectors built in 0.0s
  test: 1,000 vectors built in 0.0s



### Nearest-neighbor search: Word2Vec-averaged and FastText-averaged

The same vectorized cosine-similarity search, applied to both representations
and both evaluation sets.


In [14]:

def nearest_neighbor_search(query_vecs, train_vecs):
    """Returns (best_train_index, best_similarity) for every row in query_vecs,
    against every row in train_vecs, via one L2-normalize + matmul."""
    q_norm = l2_normalize_rows(query_vecs)
    t_norm = l2_normalize_rows(train_vecs)
    sims = q_norm @ t_norm.T          # (n_queries, n_train)
    best_idx = np.argmax(sims, axis=1)
    best_sim = sims[np.arange(len(query_vecs)), best_idx]
    return best_idx, best_sim

results = {}
for rep_name, train_vecs, val_vecs, test_vecs in [
    ("Word2Vec-avg", train_w2v_vecs, val_w2v_vecs, test_w2v_vecs),
    ("FastText-avg", train_ft_vecs, val_ft_vecs, test_ft_vecs),
]:
    t0 = time.time()
    val_idx, val_sim = nearest_neighbor_search(val_vecs, train_vecs)
    test_idx, test_sim = nearest_neighbor_search(test_vecs, train_vecs)
    print(f"{rep_name}: searched {len(val_vecs)+len(test_vecs):,} queries against "
          f"{len(train_vecs):,} training vectors in {time.time()-t0:.1f}s")
    results[rep_name] = {
        "val_idx": val_idx, "val_sim": val_sim,
        "test_idx": test_idx, "test_sim": test_sim,
    }


Word2Vec-avg: searched 2,000 queries against 1,000,000 training vectors in 3.6s
FastText-avg: searched 2,000 queries against 1,000,000 training vectors in 2.3s



### Nearest-neighbor search: Doc2Vec

Doc2Vec's own `model.dv.most_similar` already implements an optimized search
over every trained document vector, so it is used directly instead of
re-implementing the matrix-multiply search by hand.


In [15]:

def doc2vec_nearest_neighbors(token_lists, d2v_model):
    idxs, sims = [], []
    for toks in token_lists:
        inferred = d2v_model.infer_vector(toks)
        best_id, best_sim = d2v_model.dv.most_similar([inferred], topn=1)[0]
        idxs.append(best_id)
        sims.append(best_sim)
    return np.array(idxs), np.array(sims, dtype=np.float32)

print("Doc2Vec nearest-neighbor search (validation)...")
t0 = time.time()
d2v_val_idx, d2v_val_sim = doc2vec_nearest_neighbors(val_tokens, d2v_model)
print(f"  {len(val_tokens):,} validation sentences in {time.time()-t0:.1f}s")

print("Doc2Vec nearest-neighbor search (test)...")
t0 = time.time()
d2v_test_idx, d2v_test_sim = doc2vec_nearest_neighbors(test_tokens, d2v_model)
print(f"  {len(test_tokens):,} test sentences in {time.time()-t0:.1f}s")

results["Doc2Vec"] = {
    "val_idx": d2v_val_idx, "val_sim": d2v_val_sim,
    "test_idx": d2v_test_idx, "test_sim": d2v_test_sim,
}


Doc2Vec nearest-neighbor search (validation)...
  1,000 validation sentences in 16.5s
Doc2Vec nearest-neighbor search (test)...
  1,000 test sentences in 16.4s



### Qualitative examples

The first 5 validation sentences, and their nearest training-set match under
each of the three representations.


In [16]:

N_EXAMPLES = 5
for i in range(N_EXAMPLES):
    print(f"Validation[{i}]: {val_sentences_raw[i]}")
    for rep_name in ["Word2Vec-avg", "FastText-avg", "Doc2Vec"]:
        idx = results[rep_name]["val_idx"][i]
        sim = results[rep_name]["val_sim"][i]
        print(f"  [{rep_name:<13}] (sim={sim:.3f}) nearest train[{idx}]: {train_sentences_raw[idx]}")
    print()


Validation[0]: वहीं हरियाणा के जैवलिन थ्रोअर रणजीत सिंह का भी मुकाबला है।
  [Word2Vec-avg ] (sim=0.891) nearest train[356117]: पांडे का मुकाबला कांग्रेस के भोलाराम साहू से है।
  [FastText-avg ] (sim=0.942) nearest train[220153]: आईटी सेल के पंकज सिंह और आसिफ जावेद का भी इसमें योगदान है।
  [Doc2Vec      ] (sim=0.758) nearest train[477949]: इसके बाद क्वार्टर फाइनल का मुकाबला छत्तीसगढ़ और मणिपुर के बीच खेला गया, जिसमें छत्तीसगढ़ ने 2-0 से मुकाबला जीत लिया।

Validation[1]: गुंडरदेही विधानसभा आरके राय (कांग्रेस)
  [Word2Vec-avg ] (sim=0.805) nearest train[718801]: इसको मोदीनगर विधायक सुदेश शर्मा ने विधानसभा में उठाया।
  [FastText-avg ] (sim=0.891) nearest train[274129]: वहीं भाजपा नगीना लोकसभा सांसद डॉ.
  [Doc2Vec      ] (sim=0.636) nearest train[335221]: गौरतलब है कि लोकसभा में यह...

Validation[2]: कई डॉक्टर इसके चपेट में आ गए हंै।
  [Word2Vec-avg ] (sim=0.851) nearest train[757308]: कई जानलेवा बीमारियों के चपेट में आ जाते हैं।
  [FastText-avg ] (sim=0.905) nearest train[757308]: कई जानलेवा


### Aggregate comparison

Mean nearest-neighbor cosine similarity across *every* validation and test
sentence, for each representation — a single summary number per method,
alongside the qualitative examples above.


In [17]:

summary_rows = []
for rep_name in ["Word2Vec-avg", "FastText-avg", "Doc2Vec"]:
    summary_rows.append({
        "Representation": rep_name,
        "Mean NN similarity (validation)": results[rep_name]["val_sim"].mean(),
        "Mean NN similarity (test)": results[rep_name]["test_sim"].mean(),
    })
summary_df = pd.DataFrame(summary_rows)
summary_df


,Representation,Mean NN similarity (validation),Mean NN similarity (test)
0,Word2Vec-avg,0.897827,0.896562
1,FastText-avg,0.923461,0.921115
2,Doc2Vec,0.693249,0.700407



## Section 9 — Discussion

Guidance for what to look for, grounded in what each representation is built
to do — fill this in with the *actual* numbers from your run:

- **FastText vs. Word2Vec on out-of-vocabulary words:** Section 5 already
  demonstrated Word2Vec's `KeyError` on a word FastText could still embed. In
  the nearest-neighbor results, watch whether FastText's mean similarity is
  noticeably higher than Word2Vec's, especially if you spot validation/test
  sentences containing rare inflected forms — that's this exact mechanism
  showing up in the aggregate numbers, not a coincidence.
- **Averaging discards word order; Doc2Vec doesn't (directly).** A sentence
  vector built by averaging is the same regardless of word order ("dog bites
  man" and "man bites dog" would average to an identical vector). Doc2Vec's
  PV-DM objective conditions on a *sliding window* of nearby words when
  predicting each token, so word order and local structure has more chance to
  influence the learned vector. Whether that produces *better* nearest-neighbor
  matches on your actual data is exactly what the mean-similarity table above
  should show — report it, don't assume it.
- **Every representation here is unsupervised and trained only once** — the
  nearest-neighbor task is a way of *probing* embedding quality, not a
  contest with a single correct winner. A sensible write-up reports which
  representation scored highest, but also spot-checks a few of the "qualitative
  examples" above: does the retrieved sentence actually look topically or
  semantically related to the query, or just superficially similar (e.g. same
  length, common function words)? Cosine similarity alone can't tell you that
  — reading the pairs can.
- **K-means cluster quality (Section 7)** is worth a qualitative pass too: do
  the 20-nearest-to-centroid words for a given cluster share an obvious theme
  (a topic, a part of speech, a semantic field)? Note a couple of your
  clearest and murkiest clusters by way of example.
